# Basic SFT Example

This notebook runs a basic supervised fine-tuning training run against the model `Qwen/Qwen3-0.6B-Base` on the dataset `openai/gsm8k`. The dataset includes 8.5K grade school math questions and answers. The notebook is intended to load the dataset, evaluate and train the model, and review the results. For more information on preparing your environment, check out the `README.md` in the root of the repository.

<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/notebooks/basic_sft_example.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Google Colab prep

If you are running this notebook in Google Colab, be sure to select a **T4 instance**, and run the code block below to install the necessary packages.

In [ ]:
%pip install datasets numpy transformers trl tqdm huggingface_hub

## Data loading and model selection

This first cell downloads the GSM8K dataset and keeps the train and test splits we will use for fine-tuning and evaluation.

In [ ]:
from datasets import load_dataset

MODEL_ID = "Qwen/Qwen3-0.6B-Base"
DS_NAME = "openai/gsm8k"
DS_CONFIG = "main"

full_ds = load_dataset(DS_NAME, DS_CONFIG)

In [ ]:
len(full_ds["train"]), len(full_ds["test"])

The next cell sets up the model we want to train, loading both the model and the tokenizer.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
print(model.config)

print(type(model))

## Data formatting

This section turns each GSM8K example into a chat-style prompt/response pair so the model sees a natural instruction-following format during training.

In [ ]:
def format_gsm8k_example(example: dict) -> dict:
    question = example["question"].strip()
    answer = example["answer"].strip()
    return {
        "messages": [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ],
    }


train_ds = full_ds["train"].map(
    format_gsm8k_example,
    remove_columns=full_ds["train"].column_names,
)

eval_ds = full_ds["test"].map(
    format_gsm8k_example,
    remove_columns=full_ds["test"].column_names,
)

## Model evaluation

Before we perform training, we should see how well the original model does with the task in GSM8K. The following code block will evaluate performance over 100 of the questions in the `test` slice of the dataset.

In [ ]:
import re
from tqdm.auto import tqdm


def build_prompt(tokenizer, question: str) -> str:
    # Match the chat format the model was fine-tuned on.
    messages = [{"role": "user", "content": question}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def extract_gold_answer(answer: str) -> str:
    # GSM8K gold answers end with a "#### <final answer>" marker.
    marker = answer.rsplit("####", 1)
    text = marker[1] if len(marker) > 1 else answer
    match = re.search(r"-?\d[\d,]*(?:\.\d+)?", text)
    return match.group().replace(",", "") if match else ""


def extract_predicted_answer(response: str) -> str:
    # Use the last number the model produced as its final answer.
    matches = re.findall(r"-?\d[\d,]*(?:\.\d+)?", response)
    return matches[-1].replace(",", "") if matches else ""


def evaluate_model(model, tokenizer, questions, answers, desc="Evaluating", batch_size=8, max_new_tokens=256):
    correct = 0
    total = len(questions)

    was_training = model.training
    prev_use_cache = getattr(model.config, "use_cache", None)
    prev_padding_side = tokenizer.padding_side
    model.eval()
    model.config.use_cache = True
    # Decoder-only models must be left-padded for correct batched generation.
    tokenizer.padding_side = "left"

    try:
        for start in tqdm(range(0, total, batch_size), desc=desc, unit="batch"):
            batch_questions = questions[start:start + batch_size]
            batch_answers = answers[start:start + batch_size]

            prompts = [build_prompt(tokenizer, q) for q in batch_questions]
            inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                )

            # Only decode the newly generated tokens, not the echoed prompt.
            generated = outputs[:, inputs["input_ids"].shape[1]:]
            responses = tokenizer.batch_decode(generated, skip_special_tokens=True)

            for response, gold in zip(responses, batch_answers):
                if extract_predicted_answer(response) == extract_gold_answer(gold):
                    correct += 1
    finally:
        tokenizer.padding_side = prev_padding_side
        if prev_use_cache is not None:
            model.config.use_cache = prev_use_cache
        if was_training:
            model.train()

    acc = correct / total if total else 0.0
    print(f"{desc}: {correct}/{total} correct ({acc:.2%})")
    return acc

questions = [ex["question"] for ex in full_ds["test"].select(range(100))]
answers = [ex["answer"] for ex in full_ds["test"].select(range(100))]

baseline_acc = evaluate_model(model, tokenizer, questions, answers, desc="Baseline evaluation")

print(f"Baseline accuracy: {baseline_acc:.2%}")

## Training configuration

This section sets the trainer arguments, including the batch size, learning rate, mixed precision settings, and output directory. You can adjust the `SFTConfig` settings to see how they impact training time and performance.

In [ ]:
from pathlib import Path
import random

import numpy as np
from trl import SFTConfig, SFTTrainer

REPO_ROOT = Path.cwd().resolve().parent
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
RUN_NAME = "sft-gsm8k"
RUN_DIR = ARTIFACTS_DIR / RUN_NAME

if torch.cuda.is_available():
    bf16 = torch.cuda.is_bf16_supported()
    fp16 = not bf16
else:
    bf16 = False
    fp16 = False

config = SFTConfig(
    max_length=512,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    learning_rate=2e-5,
    bf16=bf16,
    fp16=fp16,
    output_dir=str(RUN_DIR),
    gradient_checkpointing=True,
    gradient_accumulation_steps=1,
    lr_scheduler_type="linear",
    eval_strategy="no",
    logging_steps=100,
    completion_only_loss=True,
)

trainer = SFTTrainer(
    args=config,
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
trainer.train()

trainer.save_model(str(RUN_DIR))

## Evaluation

This final section measures the model after fine-tuning by generating answers for a small fixed sample of GSM8K questions and checking whether the final answer matches the reference.

In [ ]:
import gc

del model
del trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()      # release cached blocks to CUDA driver
    torch.cuda.ipc_collect()

In [ ]:
# Load model and tokenizer from the fine-tuned checkpoint
model = AutoModelForCausalLM.from_pretrained(str(RUN_DIR), device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(str(RUN_DIR), trust_remote_code=True)

fine_tuned_acc = evaluate_model(model, tokenizer, questions, answers, desc="Fine-tuned evaluation")

print(f"Baseline accuracy: {baseline_acc:.2%}")
print(f"Fine-tuned accuracy: {fine_tuned_acc:.2%}")
print(f"Improvement: {fine_tuned_acc - baseline_acc:.2%}")

## Upload the trained model

In the following code block, you can upload your trained model to Hugging Face. But first, you'll need to set your Hugging Face API token.

If running locally, create a `.env` file in the `notebooks` directory and populate it with the following:

```text
HF_TOKEN=YOUR_TOKEN_VALUE
```

The `.env` file is ignored by git, so your token won't get committed to source control. For directions on generating a token, check the [documentation here](https://huggingface.co/docs/hub/security-tokens) on Hugging Face. You will need to create a token with `write` access.

If running on Google Colab, pull up the terminal and set the `HF_TOKEN` environment variable manually.

In [ ]:
# Execute this code block for local runs only
import os

for raw_line in open(".env"):
    line = raw_line.strip()
    key, value = line.split("=", 1)
    os.environ[key] = value


In [ ]:
from huggingface_hub import HfApi

# Push the fine-tuned model to the Hugging Face Hub
model.push_to_hub(RUN_NAME)
tokenizer.push_to_hub(RUN_NAME)

# Push the README.md file to the Hugging Face Hub
api = HfApi()
info = api.whoami()

api.upload_file(
    path_or_fileobj=str(RUN_DIR / "README.md"),
    path_in_repo="README.md",
    repo_id=f"{info['name']}/{RUN_NAME}",
    repo_type="model",
)

## Release used resources

The last code block releases the resources being used by the model.

In [ ]:
del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()      # release cached blocks to CUDA driver
    torch.cuda.ipc_collect()